In [ ]:
import pandas as pd
import numpy as np
import pickle  
import joblib 
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import KNNImputer, SimpleImputer

from sklearn.pipeline import Pipeline
import lightgbm as lgb
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder

import optuna
from optuna.samplers import TPESampler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, GridSearchCV
from sklearn.metrics import confusion_matrix, make_scorer, classification_report, log_loss

import warnings
warnings.filterwarnings('ignore')
# from mrmr import mrmr_classif


In [ ]:
df_train = pd.read_csv('/kaggle/input/data-science-challenge-predicting-stock-trends/training_data.csv', sep = ';', keep_default_na=False, na_values=['NA'])
df_test = pd.read_csv('/kaggle/input/data-science-challenge-predicting-stock-trends/test_data_no_target.csv', sep = ';', keep_default_na=False, na_values=['NA'])

In [ ]:
# replacing commas on floating points
df_train = df_train.replace(',', '.', regex=True)
df_test = df_test.replace(',', '.', regex=True)

In [ ]:
# Counting empty values and N/A values

print('Train Set \n')
count_of_NA_string = df_train.isin(['NA']).sum().sum()
print('count_of_NA_string:', count_of_NA_string)

count_of_empty_string = df_train.isin(['']).sum().sum()
print('count_of_empty_string:', count_of_empty_string)

count_of_NaN = df_train.isna().sum().sum()
print('count_of_NaN:', count_of_NaN)

print('')
print('Test Set \n')
count_of_NA_string = df_test.isin(['NA']).sum().sum()
print('count_of_NA_string:', count_of_NA_string)

count_of_empty_string = df_test.isin(['']).sum().sum()
print('count_of_empty_string:', count_of_empty_string)

count_of_NaN = df_test.isna().sum().sum()
print('count_of_NaN:', count_of_NaN)

In [ ]:
# check distribution of groups
df_train['Group'].value_counts()

## Handling empty strings

In [ ]:
# List of columns where 'Non-Applicable' should be replaced with 0
columns_to_zero = []

# Iterate over the columns
for column in df_train.columns:
    # Check if the column contains 'Non-Applicable'
    if '' in df_train[column].values:
        # Append the column to the list
        columns_to_zero.append(column)

print(len(columns_to_zero))

# Replace 'Non-Applicable' with 0 in the specified columns
df_train[columns_to_zero] = df_train[columns_to_zero].replace('', 0)

In [ ]:
columns_to_zero = []

# Iterate over the columns
for column in df_test.columns:
    # Check if the column contains 'Non-Applicable'
    if '' in df_test[column].values:
        # Append the column to the list
        columns_to_zero.append(column)

print(len(columns_to_zero))

# Replace 'Non-Applicable' with 0 in the specified columns
df_test[columns_to_zero] = df_test[columns_to_zero].replace('', 0)

## Data preprocessing

In [ ]:
# converting object to float data type

# changing data types to float
df_train = pd.concat([df_train.iloc[:,0], df_train[df_train.columns[1:]].astype(float)], axis=1)
df_test = pd.concat([df_test.iloc[:,0], df_test[df_test.columns[1:]].astype(float)], axis=1)

In [ ]:
#changing paramters in train set
df_train['Profatibility_comopiste'] = df_train[['I1','I2','I3','I4']].sum(axis = 1)/4
df_train['Liquidity_coposite'] = df_train[['I50','I51','I53']].sum(axis = 1)/3
df_train['Leverage_composite'] = df_train[['I54','I55','I56']].sum(axis = 1)/3
df_train['Operational_efficiency'] = df_train[['I22','I23','I24','I25','I26']].sum(axis = 1)/5
df_train['Validation_composite'] = df_train[['I39','I40','I41','I42','I43']].sum(axis = 1)/5

df_train['yr_Profatibility_comopiste'] = df_train[['dI1','dI2','dI3','dI4']].sum(axis = 1)/4
df_train['yr_Liquidity_coposite'] = df_train[['dI50','dI51','dI53']].sum(axis = 1)/3
df_train['yr_Leverage_composite'] = df_train[['dI54','dI55','dI56']].sum(axis = 1)/3
df_train['yr_Operational_efficiency'] = df_train[['dI22','dI23','dI24','dI25','dI26']].sum(axis = 1)/5
df_train['yr_Validation_composite'] = df_train[['dI39','dI40','dI41','dI42','dI43']].sum(axis = 1)/5

columns_to_remove = ['I1','I2','I3','I4','I50','I51','I53','I54','I55','I56','I22','I23','I24','I25','I26','I39','I40','I41','I42','I43','dI1','dI2','dI3','dI4',
                      'dI50','dI51','dI53','dI54','dI55','dI56','dI22','dI23','dI24','dI25','dI26','dI39','dI40','dI41','dI42','dI43']
df_train.drop(columns=columns_to_remove, inplace=True)

#changing paramters in test set
df_test['Profatibility_comopiste'] = df_test[['I1','I2','I3','I4']].sum(axis = 1)/4
df_test['Liquidity_coposite'] = df_test[['I50','I51','I53']].sum(axis = 1)/3
df_test['Leverage_composite'] = df_test[['I54','I55','I56']].sum(axis = 1)/3
df_test['Operational_efficiency'] = df_test[['I22','I23','I24','I25','I26']].sum(axis = 1)/5
df_test['Validation_composite'] = df_test[['I39','I40','I41','I42','I43']].sum(axis = 1)/5

df_test['yr_Profatibility_comopiste'] = df_test[['dI1','dI2','dI3','dI4']].sum(axis = 1)/4
df_test['yr_Liquidity_coposite'] = df_test[['dI50','dI51','dI53']].sum(axis = 1)/3
df_test['yr_Leverage_composite'] = df_test[['dI54','dI55','dI56']].sum(axis = 1)/3
df_test['yr_Operational_efficiency'] = df_test[['dI22','dI23','dI24','dI25','dI26']].sum(axis = 1)/5
df_test['yr_Validation_composite'] = df_test[['dI39','dI40','dI41','dI42','dI43']].sum(axis = 1)/5

columns_to_remove = ['I1','I2','I3','I4','I50','I51','I53','I54','I55','I56','I22','I23','I24','I25','I26','I39','I40','I41','I42','I43','dI1','dI2','dI3','dI4',
                      'dI50','dI51','dI53','dI54','dI55','dI56','dI22','dI23','dI24','dI25','dI26','dI39','dI40','dI41','dI42','dI43']
df_test.drop(columns=columns_to_remove, inplace=True)

#train and test set
display(df_train.head(2))
display(df_train.shape)


display(df_test.head(2))
display(df_test.shape)

## Drop columns with missing values > 5%

In [ ]:
drop_var = 'Perform'
target_var = 'Class'
df_train = df_train.drop(['Perform'], axis=1)
df_train.shape

In [ ]:
# col_with_missing_vals = []
col_with_major_missing_vals = []

# summarize the number of rows with missing values for each column
for i in range(df_train.shape[1]):
    # count number of rows with missing values
    n_miss = df_train.iloc[:,i].isna().sum()
    perc = n_miss / df_train.shape[0] * 100
#     print('> %d, %s Missing: %d (%.2f%%)' % (i, train_data.columns[i], n_miss, perc))
    if perc >= 5.00:
        col_with_major_missing_vals.append(df_train.columns[i])
#     elif perc <5.00 and perc >0.00:
#         col_with_minor_missing_vals.append(df_train_feature.columns[i])
# print(f'{len(col_with_missing_vals)} out of {len(df_train_feature.columns)} have missing values')
print(f'{len(col_with_major_missing_vals)} out of {len(df_train.columns)} have missing values > 5%')

In [ ]:
col_with_major_missing_vals

In [ ]:
df_train = df_train.drop(col_with_major_missing_vals, axis=1)

## Creating train test sets

In [ ]:
print("target_var: ", target_var)
X = df_train.drop([target_var, 'Group'], axis=1)
# X = train_data[corr_features+calculated_columns]
y = df_train[target_var]
print(X.shape, y.shape)

In [ ]:
# Splitting train and set data
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.80, random_state=42, stratify=y)
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

## Util functions

In [ ]:
def result(model, X_val, y_val, model_key=None):
    # Check if label adjustment is needed (for models like XGBoost)
    if model_key == 'xgb':
        # Adjust labels to be non-negative
        y_train = y_train + 1  # Mapping -1 to 0, 0 to 1, and 1 to 2
        y_val = y_val + 1      # Same mapping for validation labels

#     # Fit the model
#     model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    # Calculate cross-validation scores
#     try:
#         scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1, error_score='raise')
#         print(f'Mean CV score: {np.mean(scores):.2f}, Std: {np.std(scores):.2f}')
#     except Exception as e:
#         print(f"Error calculating cross-validation scores: {e}")

    # Print model accuracy on the validation set
    print(f'Model score on validation set: {model.score(X_val, y_val):.2f}')

    # Print classification report
    print(classification_report(y_val, y_pred))

    # Dynamic labels for the confusion matrix
    labels = sorted(np.unique(np.concatenate((y_val, y_pred))))
    sns.heatmap(confusion_matrix(y_val, y_pred), annot=True, fmt="d", cmap="Blues", 
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted labels')
    plt.ylabel('Actual labels')
    plt.title('Confusion Matrix')
    plt.show()

## Model Training

### HistGradientBoostingClassifier

In [ ]:
hgb = HistGradientBoostingClassifier()

result(hgb, X_train, y_train, X_val, y_val, cv=1)

In [ ]:
X_test = df_test.drop(col_with_major_missing_vals+['Group'], axis=1)
# X_test = imputer.transform(X_test)

test_preds = hgb.predict(X_test).astype(int)
print(test_preds[:5])
print(test_preds.shape)

In [ ]:
output_file = "submission_hgb.txt"

# Open the file in write mode
with open(output_file, "w") as file:
    # Write each prediction to the file
    for prediction in test_preds:
        file.write(str(prediction) + "\n")

print("Predictions exported to:", output_file)

### Optimize hgb

In [ ]:
def objective(trial, X_train, y_train, X_test, y_test):
        
    # Suggest values for the hyperparameters
    params = {
        'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.2),
        'max_iter': trial.suggest_int("max_iter", 100, 1000),
        'max_depth': trial.suggest_int("max_depth", 3, 10),
        'min_samples_leaf': trial.suggest_int("min_samples_leaf", 10, 50),
        'max_bins': trial.suggest_int("max_bins", 100, 255),
        'l2_regularization': trial.suggest_float("l2_regularization", 0, 1)
    }
    
    # Create a HistGradientBoostingClassifier with suggested hyperparameters
    classifier = HistGradientBoostingClassifier(**params, random_state=42)
    
    # Train the model
    classifier.fit(X_train, y_train)
    
    # Evaluate the model
    y_pred = classifier.predict(X_test)
    
    # Define cost matrix
    cost_matrix = np.array([[0, 1, 2], [1, 0, 1], [2, 1, 0]])
    
    # Calculate custom error score
    error_score = np.sum(confusion_matrix(y_test, y_pred) * cost_matrix) / len(y_test)
    
    return error_score

In [ ]:
# sampler for Optuna optimization
sampler = TPESampler(seed=42)  # Using Tree-structured Parzen Estimator sampler for optimization

# Create a study object
study = optuna.create_study(direction="minimize", sampler=sampler)  # Note the direction change to minimize for error scorer

# Run the optimization process
study.optimize(lambda trial: objective(trial, X_train, y_train, X_val, y_val), n_trials=25)

# best parameters after optimization
best_params = study.best_params

print('='*20)
print(best_params)

In [ ]:
# best_params = {'learning_rate': 0.1957650976029255, 
#                'max_iter': 662, 
#                'max_depth': 10, 
#                'min_samples_leaf': 26, 
#                'max_bins': 204, 
#                'l2_regularization': 0.00922737312841293}

# Definingmodel
model = HistGradientBoostingClassifier()

# Define cost matrix
cost_matrix = np.array([[0, 1, 2], [1, 0, 1], [2, 1, 0]])

error_scorer = make_scorer(lambda y_true, y_pred: np.sum(confusion_matrix(y_true, y_pred) * cost_matrix) / len(y_true), greater_is_better=False)

best_params = {'learning_rate': [0.040201910182771604, 0.1957650976029255, 0.010204721574458758], 
                'max_iter': [124, 160, 662], 
                'max_depth': [6, 9, 10],
                'min_samples_leaf': [21, 26, 29], 
                'max_bins': [154, 166, 254], 
                'l2_regularization': [0.00922737312841293, 0.4523676205795673, 0.7794432417755719]
                }

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=best_params, cv=3, scoring=error_scorer)

# Perform grid search
grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_estimator_
print("Best parameters:", best_params)



In [ ]:
params = {'learning_rate': 0.010204721574458758, 'max_iter': 124, 'max_depth': 9, 'min_samples_leaf': 21, 'max_bins': 166, 'l2_regularization': 0.7794432417755719}

model = HistGradientBoostingClassifier(**params)
# Perform grid search
model.fit(X_train, y_train)

# Evaluate the best model on the test set
y_pred = model.predict(X_val)

# Confusion matrix
cfm = confusion_matrix(y_val, y_pred)
print('Confusion matrix:\n', cfm)

# Calculate error
err = np.sum(cfm * cost_matrix) / len(y_val)
print("Error:", err)

# Classification report
print("Classification Report:\n", classification_report(y_val, y_pred))

# # Calculate error
# err = np.sum(cfm * cost_matrix) / len(y_test)
# print("Error:", err)

# result(model_hgb, X_train, y_train, X_val, y_val, cv=3)

In [ ]:
X_test = df_test.drop(col_with_major_missing_vals+['Group'], axis=1)
# X_test = imputer_median.transform(X_test)

test_preds = model.predict(X_test).astype(int)
print(test_preds[:5])
print(test_preds.shape)

In [ ]:
output_file = "submission_hgb_4.txt"

# Open the file in write mode
with open(output_file, "w") as file:
    # Write each prediction to the file
    for prediction in test_preds:
        file.write(str(prediction) + "\n")

print("Predictions exported to:", output_file)

### Random Forest

In [ ]:
cols = X_train.columns
model_rf = RandomForestClassifier()

imputer_knn = KNNImputer(n_neighbors=3, weights='uniform', metric='nan_euclidean')

X_train = pd.DataFrame(imputer_knn.fit_transform(X_train), columns=cols)
X_val = pd.DataFrame(imputer_knn.transform(X_val), columns=cols)

In [ ]:
# Define cost matrix
cost_matrix = np.array([[0, 1, 2], [1, 0, 1], [2, 1, 0]])
error_scorer = make_scorer(lambda y_true, y_pred: np.sum(confusion_matrix(y_true, y_pred) * cost_matrix) / len(y_true), greater_is_better=False)

# Perform grid search
model_rf.fit(X_train, y_train)

# Evaluate the best model on the test set
y_pred = model_rf.predict(X_val)

# Confusion matrix
cfm = confusion_matrix(y_val, y_pred)
print('Confusion matrix:\n', cfm)

# Calculate error
err = np.sum(cfm * cost_matrix) / len(y_val)
print("Error:", err)

# Classification report
print("Classification Report:\n", classification_report(y_val, y_pred))

In [ ]:
X_test = df_test.drop(col_with_major_missing_vals+['Group'], axis=1)
X_test = imputer_median.transform(X_test)

test_preds = model_rf.predict(X_test).astype(int)
print(test_preds[:5])
print(test_preds.shape)

In [ ]:
output_file = "submission_rf_11.txt"

# Open the file in write mode
with open(output_file, "w") as file:
    # Write each prediction to the file
    for prediction in test_preds:
        file.write(str(prediction) + "\n")

print("Predictions exported to:", output_file)